# SAQR-100M — One-click real-data training
This notebook downloads the real Arabic E-Book Corpus automatically, prepares the text, trains SAQR-100M from random initialization, and generates a first response.


In [ ]:
!git clone -q https://github.com/majsjsj/saqr-ai.git
%cd saqr-ai
!pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
import subprocess, os

url = 'https://api.researchdata.se/dataset/2024-145/1/file/zip'
Path('data/hindawi').mkdir(parents=True, exist_ok=True)
print('Downloading the real Arabic E-Book Corpus...')
subprocess.run(['curl','-L','--fail','--retry','3','--retry-delay','2',url,'-o','/tmp/hindawi.zip'], check=True)
print('Extracting...')
subprocess.run(['unzip','-q','-o','/tmp/hindawi.zip','-d','data/hindawi'], check=True)

txt_files = [p for p in Path('data/hindawi').rglob('*') if p.is_file() and p.suffix.lower()=='.txt']
print('TXT files found:', len(txt_files))
if not txt_files:
    raise RuntimeError('No TXT files found in the downloaded corpus.')

with open('data/train.txt','w',encoding='utf-8') as out:
    for p in sorted(txt_files):
        try:
            text = p.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if text.strip():
            out.write('\n\n')
            out.write(text)

size_gb = Path('data/train.txt').stat().st_size / (1024**3)
print(f'Corpus ready: {size_gb:.2f} GiB')


In [ ]:
!python tokenizer.py --input data/train.txt --output tokenizer --vocab-size 32000


In [ ]:
!python train.py --config configs/saqr_100m.json --data data/train.txt --tokenizer tokenizer/tokenizer.json


In [ ]:
!python generate.py --checkpoint checkpoints/last.pt --tokenizer tokenizer/tokenizer.json --prompt "أنت SAQR. من أنت؟ وما الذي تستطيع فعله؟" --tokens 120


## Important
SAQR-100M is trained from random initialization. The corpus is downloaded automatically; no manual dataset upload is required.
